# Replication of stim's DEM sampling results

This notebook aims to show how sampling results from stims surface code experiments can be replicated using paritea. To start, we construct such an experiment and import it into paritea, producing:
- The diagram itself
- The noise model (i.e. the faults) that stim inserted on the diagrams edges
- The ids of nodes belonging to measurements (both in-cycle and out-cycle)
- A map from observable indices to the Pauli string representing it (X on the edge incident to the included measurement if the measurement was in the X basis, otherwise Z)
- A list of detectors through their representing Pauli strings (see previous)

Note that stim circuit qubits are always assumed to be initialized in the zero state (by design) and measured out at the end (by paritea assumptions). This means that the importer cannot handle state preparations without the user adding (ignored) measurements to the output.

In [ ]:
import stim

from paritea.glue.stim import from_stim

p = 1e-3
c = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=2,
    distance=7,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
    before_measure_flip_probability=p,
    before_round_data_depolarization=p,
)
c = c.flattened()
d, nm, measurement_nodes, observables, detectors = from_stim(c)

With this information, paritea can extract detecting regions (and only detecting regions, since the diagram is closed), and find those regions that anticommute with the given Pauli strings from the import phase. The noise model is subsequently pushed out of the diagram, with anticommutation with regions recorded in the faults, and equivalent faults are compressed by combining their recorded probabilities.

At this point the noise model is essentially a stim DEM, without disambiguation between regions corresponding to observables / detectors. Information required for this distinction is obtained during push-out, so the export can proceed with it.

In [ ]:
import sinter

from paritea.glue.stim import export_to_stim_dem, push_out_for_measurement_detectors, wrap_dem_as_sinter_task

pushed_out, logical_regions, detector_regions = push_out_for_measurement_detectors(
    nm,
    measurement_nodes=measurement_nodes,
    logicals=list(observables.values()),
    detectors=detectors,
)
pushed_out.compress(lambda x, y: x * (1 - y) + (1 - x) * y)
dem = export_to_stim_dem(
    pushed_out,
    logical_regions=logical_regions,
    detector_regions=detector_regions,
)

We can now simulate the DEM constructed internally by stim and the DEM constructed through paritea via high-speed sampling with sinter.

In [ ]:
stats = sinter.collect(
    num_workers=16,
    tasks=[
        sinter.Task(circuit=c, json_metadata={"p": p, "name": "stim"}),
        wrap_dem_as_sinter_task(dem, json_metadata={"p": p, "name": "paritea"}),
    ],
    max_shots=100_000_000,
    max_errors=1_000,
    decoders=["pymatching"],
    print_progress=True,
)

With a bit of plotting, we can indeed assess that the logical error rates are closely matching and are within their respective default confidence intervals.

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(1, 1)
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stats: stats.json_metadata["p"],
    group_func=lambda stats: stats.json_metadata["name"],
)
[stim_stats, paritea_stats] = stats
print(f"Stim: {stim_stats.errors / stim_stats.shots} error rate")
print(f"Paritea: {paritea_stats.errors / paritea_stats.shots} error rate")
ax.set_ylim(auto=True)
ax.set_xlim(auto=True)
ax.loglog()
ax.set_xlabel("Physical Error Rate")
ax.set_ylabel("Logical Error Rate per Shot")
ax.grid(which="major")
ax.grid(which="minor")
ax.legend()
fig.set_dpi(120)  # Show it bigger